# SigLIP Diagnostic & Saliency-Plateau Benchmark
This notebook investigates SigLIP inference, query templates, and probability calibration.

### Objectives:
1. **SigLIP2 Notebook Parity**: Test prompt templates from `MSSearch/vis/SigLIP2_inference.ipynb` (`The photo of {label}.` vs `a photo of {q}` vs raw tags).
2. **Calibration Diagnosis**: Compare raw cosine similarity $\cos(I, T)$, SigLIP native sigmoid $\sigma(\text{logit})$, and linear dynamic range calibration $[0.02, 0.24] \to [0.0, 1.0]$.
3. **Stage 3 Saliency Plateau**: Measure $\Delta S = S_{\text{max\_child}} - S_{\text{parent}}$ on parent vs 4 child quadtree boxes for target vs background to determine why $\tau = 0.04$ stops early with sigmoid probabilities.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, REPOSITORY CLONE & BRANCH SYNCHRONIZATION
# ==============================================================================
import os
import sys
import subprocess
import torch

TARGET_BRANCH = 'draft/improve_all'
REPO_DIR = '/kaggle/working/MSSearch' if os.path.exists('/kaggle') else os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists(os.path.join(REPO_DIR, 'mssearch')) and os.path.exists('mssearch'):
    REPO_DIR = os.path.abspath(os.getcwd())

# Retrieve GitHub Access Token if in Kaggle Secrets
github_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret('MaCVi')
    print('[SETUP] Retrieved GitHub token from Kaggle Secret.')
except Exception:
    github_token = os.environ.get('MaCVi') or os.environ.get('GITHUB_TOKEN')

if os.path.exists('/kaggle') and not os.path.exists(REPO_DIR):
    clone_url = f'https://{github_token}@github.com/tothanhdat2006/MSSearch.git' if github_token else 'https://github.com/tothanhdat2006/MSSearch.git'
    print(f'[SETUP] Cloning MSSearch repository (branch: {TARGET_BRANCH}) into {REPO_DIR}...')
    ret = subprocess.run(['git', 'clone', '-b', TARGET_BRANCH, clone_url, REPO_DIR], capture_output=True, text=True)
    if ret.returncode != 0:
        subprocess.run(['git', 'clone', clone_url, REPO_DIR], capture_output=True, text=True)
        subprocess.run(['git', '-C', REPO_DIR, 'checkout', TARGET_BRANCH], capture_output=True, text=True)
elif os.path.exists(REPO_DIR):
    print(f'[SETUP] Updating repository at {REPO_DIR} to branch {TARGET_BRANCH}...')
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', TARGET_BRANCH], capture_output=True, text=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', TARGET_BRANCH], capture_output=True, text=True)

# Ensure REPO_DIR and pilot directories are on sys.path
for p in [REPO_DIR, os.path.join(REPO_DIR, 'pilot'), os.path.join(REPO_DIR, 'pilot', 'multisource_hrbench')]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'[ENVIRONMENT] Device: {device} | PyTorch: {torch.__version__}')


In [ ]:
# ==============================================================================
# CELL 2: RUN COMPREHENSIVE SIGLIP DIAGNOSTIC
# Evaluates Synthetic crops and HRBench Sample 0 crops side-by-side
# ==============================================================================
import os
import sys

# Locate debug_siglip.py dynamically
script_path = None
candidates = [
    os.path.join(REPO_DIR, 'pilot', 'multisource_hrbench', 'debug_siglip.py'),
    os.path.join(os.getcwd(), 'pilot', 'multisource_hrbench', 'debug_siglip.py'),
    os.path.join(os.getcwd(), 'debug_siglip.py'),
]
for c in candidates:
    if os.path.exists(c):
        script_path = c
        break

if script_path is None:
    for root, _, files in os.walk(REPO_DIR if os.path.exists(REPO_DIR) else os.getcwd()):
        if 'debug_siglip.py' in files:
            script_path = os.path.join(root, 'debug_siglip.py')
            break

print(f'[RUNNER] Located script at: {script_path}')
assert script_path is not None and os.path.exists(script_path), f'Cannot locate debug_siglip.py in {REPO_DIR}'

script_dir = os.path.dirname(script_path)
if script_dir not in sys.path:
    sys.path.insert(0, script_dir)

try:
    from debug_siglip import main
except ImportError:
    from pilot.multisource_hrbench.debug_siglip import main

sys.argv = ['debug_siglip.py', '--sample_idx', '0']
main()
